This notebook contains the gathering of the data into a comprehensive dataset for Homework 1.

In [1]:
# Import the necessary libraries
import os
import tabulate
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pyarrow
import sportsdataverse as sdv
import polars as pl
from great_tables import GT, md
from adjustText import adjust_text

In [ ]:
# Load up play by play data and game data for the 2021-2025 seasons
# from the sportsdataverse library
plays_df_polars = sdv.cfb.load_cfb_pbp(seasons=[2021, 2022, 2023, 2024, 2025], return_as_pandas=False)

# Convert the polar files to pandas dataframes
pbp_2021_2025_raw = plays_df_polars.to_pandas(use_pyarrow_extension_array=False)

In [7]:
# examine the shape of the play by play data
pbp_2021_2025_raw.shape

(778447, 477)

In [ ]:
# Examine the available columns of the play by play data
col_list = pbp_2021_2025_raw.columns.tolist()
pd.set_option('display.max_seq_items', None)
print("\n".join(pbp_2021_2025_raw.columns))

season
game_id
game_play_number
pos_team_id
pos_team
def_pos_team_id
def_pos_team
pos_team_score
def_pos_team_score
half
period
down
distance
EPA
wpa
wp_before
wp_after
def_wp_before
def_wp_after
penalty_detail
yds_penalty
penalty_1st_conv
def_EPA
rz_play
scoring_opp
middle_8
stuffed_run
change_of_pos_team
downs_turnover
pos_score_diff_start
pos_score_pts
home_wp_before
away_wp_before
home_wp_after
away_wp_after
end_of_half
lead_pos_team
lead_play_type
lag_pos_team
orig_play_type
offense_score_play
defense_score_play
pos_score_diff
change_of_poss
rusher_player_name
yds_rushed
passer_player_name
receiver_player_name
yds_receiving
yds_sacked
sack_players
sack_player_name
sack_player_name2
pass_breakup_player_name
interception_player_name
yds_int_return
fumble_player_name
fumble_forced_player_name
fumble_recovered_player_name
yds_fumble_return
punter_player_name
yds_punted
yds_punt_return
yds_punt_gained
punt_block_player_name
punt_block_return_player_name
fg_kicker_player_name
yds_fg
fg_

In [19]:
# explore values of the orig_play_type column
pbp_2021_2025_raw['pass'].value_counts()

pass
False    484584
True     293863
Name: count, dtype: int64

In [ ]:
# explore values of the seasonType column
pbp_2021_2025_raw['seasonType'].value_counts()

seasonType
2    739858
3     37573
4      1016
Name: count, dtype: int64

During the filtering process, I will filter out the following:

- Regular season only
- There is a valid possesion team (not na)
- The play type is a pass or a run
- epa is not null
- wpa is not null
- filter for only power 4 conferences (ACC, Big 12, Big 10, SEC)


Note: as seen above, the seasonType column has a few categories. 

According to the SportsDataVerse website: 

2 = regular season
3 = post season.

However, it does not say anything about 4. Since we will filter for 2, all 4's will be filtered out. I am quite sure that the rows with 4 are errors of some kind.

Here is the link I used to find this information:
https://py.sportsdataverse.org/docs/cfb/reference/loaders

It seems that this data does not include the conference of the school. 

Using the link above, I will also load in the schedule data for these games. Below, I will import the schedule and merge the needed columns into our play by play data. This will give us conference information for home and away teams.

In [42]:
# load in the game schedule data for the 2021-2025 seasons
schedule_df = sdv.cfb.load_cfb_schedule(seasons=[2021, 2022, 2023, 2024, 2025], return_as_pandas=False)

# convert to pandas dataframe
schedule_df = schedule_df.to_pandas(use_pyarrow_extension_array=False)

# examine the available columns of the game schedule data
print("\n".join(schedule_df.columns))

game_id
season
week
season_type
start_date
start_time_tbd
completed
neutral_site
conference_game
attendance
venue_id
venue
home_id
home_team
home_division
home_conference
home_points
home_post_win_prob
home_pregame_elo
home_postgame_elo
away_id
away_team
away_division
away_conference
away_points
away_post_win_prob
away_pregame_elo
away_postgame_elo
excitement_index
highlights
notes


In [48]:
# filter out uneeded columns from the schedule data
schedule_filtered = schedule_df[['game_id', 'week', 'season', 'home_team', 'away_team', 'home_conference', 'away_conference', 'season_type', 'home_division', 'away_division', 'home_id', 'away_id']]

Because the season_type categories are 'regular' and 'postseason'. We will need to transform those into 2's and 3's to match the categories in the pbp data.

In [51]:
# in the season_type column, we will transform regular to 2 and postseason to 3
schedule_filtered['season_type'] = schedule_filtered['season_type'].replace({'regular': 2, 'postseason': 3})

# check that the values have been transformed correctly
schedule_filtered['season_type'].value_counts()

season_type
2    17152
3      279
Name: count, dtype: int64

In [60]:
# merge the datasets together to create a single dataset for analysis
pbp_merged = pd.merge(pbp_2021_2025_raw, schedule_filtered, how='left', left_on=['game_id', 'week', 'season', 'seasonType', 'homeTeamId', 'awayTeamId'], right_on=['game_id', 'week', 'season', 'season_type', 'home_id', 'away_id'])

In [61]:
# examine the shape of the game data
pbp_merged.shape

(778447, 486)

In [62]:
# Examine the conference columns of the game data
pbp_merged[['home_conference']].value_counts()

home_conference  
SEC                  96140
Big Ten              94072
ACC                  89944
Big 12               77579
Sun Belt             70608
American Athletic    70422
Mountain West        64589
Mid-American         58729
Conference USA       58027
Pac-12               45651
FBS Independents     24366
MVFC                  5454
Big Sky               3427
CAA                   1826
Southern               319
Big South-OVC          309
Southland              172
SWAC                   159
AWC                    157
Name: count, dtype: int64

In [63]:
# Filter the data to include only regular season plays,
# valid posseion team,
# pass or run plays only,
# power 4 conferences only(ACC, Big 12, Big 10, SEC)
# and valid EPA and WPA values
pbp_filtered = pbp_merged[
    (pbp_merged['season_type'] == 2) &
    (pbp_merged['pos_team'].notna()) &
    ((pbp_merged['pass'] == True) | (pbp_merged['rush'] == True)) &
    (pbp_merged['EPA'].notna()) &
    (pbp_merged['wpa'].notna()) &
    ((pbp_merged['home_conference'].isin(['ACC', 'Big 12', 'Big Ten', 'SEC'])) |
     (pbp_merged['away_conference'].isin(['ACC', 'Big 12', 'Big Ten', 'SEC'])))
].copy()

Now, let's check up on the seasonType value to make sure.

In [71]:
# check the seasonType values in the filtered data
pbp_filtered['seasonType'].sort_values(ascending=False)

1         2
528297    2
528304    2
528303    2
528302    2
         ..
266027    2
266028    2
266029    2
266030    2
770205    2
Name: seasonType, Length: 280163, dtype: object

In [ ]:
# Check the shape of the filtered data
pbp_filtered.shape

(280163, 486)

In [72]:
pbp_filtered.columns

Index(['season', 'game_id', 'game_play_number', 'pos_team_id', 'pos_team',
       'def_pos_team_id', 'def_pos_team', 'pos_team_score',
       'def_pos_team_score', 'half', 'period', 'down', 'distance', 'EPA',
       'wpa', 'wp_before', 'wp_after', 'def_wp_before', 'def_wp_after',
       'penalty_detail', 'yds_penalty', 'penalty_1st_conv', 'def_EPA',
       'rz_play', 'scoring_opp', 'middle_8', 'stuffed_run',
       'change_of_pos_team', 'downs_turnover', 'pos_score_diff_start',
       'pos_score_pts', 'home_wp_before', 'away_wp_before', 'home_wp_after',
       'away_wp_after', 'end_of_half', 'lead_pos_team', 'lead_play_type',
       'lag_pos_team', 'orig_play_type', 'offense_score_play',
       'defense_score_play', 'pos_score_diff', 'change_of_poss',
       'rusher_player_name', 'yds_rushed', 'passer_player_name',
       'receiver_player_name', 'yds_receiving', 'yds_sacked', 'sack_players',
       'sack_player_name', 'sack_player_name2', 'pass_breakup_player_name',
       'interceptio

In [73]:
pbp_filtered.shape

(280163, 486)

Now, let's clean up the columns.